# Đánh giá và so sánh mô hình gom cụm

Notebook này tổng hợp kết quả của K-Means, Hierarchical Clustering và DBSCAN, tính ARI, trực quan hóa bằng PCA và đề xuất mô hình cuối.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "evaluation":
    PROJECT_ROOT = PROJECT_ROOT.parent

CLUSTER_RESULTS = PROJECT_ROOT / "clustering" / "results"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "evaluation" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", None)


## 1. Đọc nhãn và kết quả tinh chỉnh

In [ ]:
labels = pd.read_csv(CLUSTER_RESULTS / "labels.csv", dtype={"Customer ID": "string"})
kmeans_tuning = pd.read_csv(CLUSTER_RESULTS / "tuning_kmeans.csv")
hierarchical_tuning = pd.read_csv(CLUSTER_RESULTS / "tuning_hierarchical.csv")
dbscan_tuning = pd.read_csv(CLUSTER_RESULTS / "tuning_dbscan.csv")

print("So khach:", len(labels))
display(labels.head())


### Chọn cấu hình đã lưu

Các dòng `selected=True` do notebook TV2 đánh dấu là cấu hình cuối cho từng thuật toán.

In [ ]:
kmeans_selected = kmeans_tuning[kmeans_tuning["selected"]].iloc[0]
hierarchical_selected = hierarchical_tuning[hierarchical_tuning["selected"]].iloc[0]
dbscan_selected = dbscan_tuning[dbscan_tuning["selected"]].iloc[0]

print("K-Means:", kmeans_selected.to_dict())
print("Hierarchical:", hierarchical_selected.to_dict())
print("DBSCAN:", dbscan_selected.to_dict())


## 2. Bảng so sánh mô hình

- **Silhouette**: càng gần 1 càng tốt.
- **Davies-Bouldin**: càng nhỏ càng tốt.
- **Calinski-Harabasz**: càng lớn càng tốt.
- Với DBSCAN, các chỉ số được tính sau khi bỏ nhãn nhiễu `-1`.

In [ ]:
noise_ratio = float((labels["dbscan"] == -1).mean())

comparison = pd.DataFrame([
    {
        "Mo hinh": "K-Means",
        "Cau hinh": f"k={int(kmeans_selected['n_clusters'])}",
        "So cum": int(kmeans_selected["n_clusters"]),
        "Ty le nhieu": 0.0,
        "Silhouette": kmeans_selected["silhouette"],
        "Davies-Bouldin": kmeans_selected["davies_bouldin"],
        "Calinski-Harabasz": kmeans_selected["calinski_harabasz"],
    },
    {
        "Mo hinh": "Hierarchical",
        "Cau hinh": f"{hierarchical_selected['linkage']} - k={int(hierarchical_selected['k'])}",
        "So cum": int(hierarchical_selected["n_clusters"]),
        "Ty le nhieu": 0.0,
        "Silhouette": hierarchical_selected["silhouette"],
        "Davies-Bouldin": hierarchical_selected["davies_bouldin"],
        "Calinski-Harabasz": hierarchical_selected["calinski_harabasz"],
    },
    {
        "Mo hinh": "DBSCAN",
        "Cau hinh": f"eps={dbscan_selected['eps']:.6g}, min_samples={int(dbscan_selected['min_samples'])}",
        "So cum": int(dbscan_selected["n_clusters"]),
        "Ty le nhieu": noise_ratio,
        "Silhouette": dbscan_selected["silhouette"],
        "Davies-Bouldin": dbscan_selected["davies_bouldin"],
        "Calinski-Harabasz": dbscan_selected["calinski_harabasz"],
    },
])

display(comparison)
results_dir = PROJECT_ROOT / "evaluation" / "results"
results_dir.mkdir(parents=True, exist_ok=True)
comparison.to_csv(results_dir / "model_comparison.csv", index=False)


## 3. Độ ổn định K-Means theo nhiều `random_state`

ARI = 1 nếu hai lần phân cụm trùng khớp hoàn toàn; giá trị gần 0 nghĩa là kết quả gần với phân cụm ngẫu nhiên. Cần file `data/processed/X_scaled.csv` từ TV1.

In [ ]:
x_path = DATA_PROCESSED / "X_scaled.csv"
X = None
stability = None

if x_path.exists():
    X_df = pd.read_csv(x_path, dtype={"Customer ID": "string"}).set_index("Customer ID")
    X_df = X_df.reindex(labels["Customer ID"])
    if X_df.isna().all().all():
        raise ValueError("X_scaled.csv khong co Customer ID trung voi labels.csv")
    X = X_df.to_numpy()

    k = int(kmeans_selected["n_clusters"])
    reference_labels = labels["kmeans"].to_numpy()
    stability = []
    for seed in [0, 1, 7, 42, 100]:
        model = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=seed)
        seed_labels = model.fit_predict(X)
        stability.append({
            "random_state": seed,
            "ARI_vs_saved_labels": adjusted_rand_score(reference_labels, seed_labels),
            "inertia": model.inertia_,
        })
    stability = pd.DataFrame(stability)
    display(stability)
    stability.to_csv(results_dir / "kmeans_stability.csv", index=False)
else:
    warnings.warn(f"Khong tim thay {x_path}. Stability va PCA se duoc bo qua.")


## 4. So sánh mức độ giống nhau giữa các thuật toán

ARI được tính giữa K-Means, Hierarchical và DBSCAN. Hai dòng cuối xem riêng kết quả DBSCAN khi gồm và khi bỏ điểm nhiễu.

In [ ]:
dbscan_mask = labels["dbscan"] != -1
ari_table = pd.DataFrame([
    {"Cap mo hinh": "K-Means vs Hierarchical", "ARI": adjusted_rand_score(labels["kmeans"], labels["hierarchical"]), "Pham vi": "Toan bo khach"},
    {"Cap mo hinh": "K-Means vs DBSCAN", "ARI": adjusted_rand_score(labels["kmeans"], labels["dbscan"]), "Pham vi": "Gom nhieu -1"},
    {"Cap mo hinh": "K-Means vs DBSCAN", "ARI": adjusted_rand_score(labels.loc[dbscan_mask, "kmeans"], labels.loc[dbscan_mask, "dbscan"]), "Pham vi": "Bo nhieu -1"},
])
display(ari_table)
ari_table.to_csv(results_dir / "model_ari.csv", index=False)


## 5. PCA 2D cho từng mô hình

In [ ]:
if X is not None:
    coords = PCA(n_components=2, random_state=42).fit_transform(X)
    fig, axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)

    for ax, column, title in zip(
        axes,
        ["kmeans", "hierarchical", "dbscan"],
        ["K-Means", "Hierarchical", "DBSCAN (nhieu mau -1)"],
    ):
        for cluster in np.unique(labels[column].to_numpy()):
            mask = labels[column].to_numpy() == cluster
            ax.scatter(
                coords[mask, 0],
                coords[mask, 1],
                s=10,
                alpha=0.55,
                label=str(cluster),
                color="lightgray" if cluster == -1 else None,
            )
        ax.set(title=title, xlabel="PCA 1", ylabel="PCA 2")
        ax.legend(title="Cum", fontsize=8)

    fig.savefig(FIGURES_DIR / "pca_three_models.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("Chua tao PCA vi thieu data/processed/X_scaled.csv.")


## 6. Kết luận sơ bộ

Quyết định mô hình cuối không được dựa hoàn toàn trên một chỉ số duy nhất:

1. Ưu tiên mô hình có cụm có kích thước hợp lý và dễ diễn giải về mặt kinh doanh.
2. Chỉ so sánh các chỉ số trên cùng một không gian dữ liệu.
3. Với DBSCAN, luôn ghi rõ tỷ lệ nhiễu và đánh giá riêng nhóm ngoại biên.
4. Nếu các chỉ số mâu thuẫn, cần xem thêm độ ổn định, kích thước cụm, ARI và mức độ phân biệt R/F/M.

In [ ]:
ari_km_hier = adjusted_rand_score(labels["kmeans"], labels["hierarchical"])

decision = pd.DataFrame({
    "Tieu chi": [
        "Ty le nhieu DBSCAN",
        "ARI K-Means - Hierarchical",
        "Silhouette cao nhat",
        "Quyet dinh mo hinh cuoi",
    ],
    "Ket qua": [
        f"{noise_ratio:.2%}",
        f"{ari_km_hier:.4f}",
        comparison.loc[comparison["Silhouette"].idxmax(), "Mo hinh"],
        "Cho Task 2/cluster profiling xac nhan tinh y nghia kinh doanh",
    ],
})
display(decision)
